# SD-008 D2: Synthetic Dataset Generator

This notebook is a thin driver. All logic lives in the `sd008_gen` package next to it; here we run it, check the result, and look at the data.

**What D2 produces.** Five normalized tables (`cardholders`, `cards`, `merchants`, `transactions`, `fraud_labels`) plus an aggregate `label_gap_summary.csv` that compares true fraud with what investigators actually observed. Hidden ("latent") variables such as compromise states, travel states and merchant vulnerability are used internally but are never written to the exported tables.

**How to run.** Set `SCALE` below to `"test"` (small, fast) or `"target"` (the agreed size) and run all cells. Output goes to `deliverables/D2_Synthetic_Dataset_Generator/outputs/` (gitignored). Same seed and config give the same files.

## 1. Setup
Find the `sd008_gen` package (works from the repo root or the D2 folder), load the approved scenario config, and show its status and seed.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

SCALE = "test"   # must be exactly "test" (2,000 holders) or "target" (5,000 holders)
if SCALE not in ("test", "target"):
    raise ValueError(f'SCALE is {SCALE!r}; set it to exactly "test" or "target" in this cell and run again.')

here = Path.cwd().resolve()
D2_DIR = next(p for p in [here, *here.parents, *(c for c in here.glob("deliverables/*"))] if (p / "sd008_gen").is_dir())
sys.path.insert(0, str(D2_DIR))
OUTPUT_ROOT = D2_DIR / "outputs"

from sd008_gen.config import load_config
from sd008_gen.generate import generate
from sd008_gen.export import config_sha256, non_key_features, TABLES
from sd008_gen.validate import validate_all

cfg = load_config(D2_DIR / "config" / "scenario_v0.yaml")
print("status:", cfg["meta"]["status"])
print("seed  :", cfg["simulation"]["seed"], "| scale:", SCALE)

## 2. Generate
Runs the full pipeline. If a run folder for this seed and config already exists, nothing is deleted or overwritten: the existing CSVs are loaded instead (the data is regenerated in memory only, because the generator-side diagnostics need internal truth that is never saved).

In [ ]:
import json, time
cfg_run = {**cfg, "simulation": {**cfg["simulation"], **(cfg["simulation"]["target_scale"] if SCALE == "target" else {})}}
run_dir = OUTPUT_ROOT / f"run_{cfg['simulation']['seed']}_{config_sha256(cfg_run)[:8]}"
t0 = time.time()
if run_dir.exists():
    print("Run directory already exists, reusing CSVs from:", run_dir)
    res = generate(cfg, out_root=None, scale=SCALE, return_internal=True)
    res["run_dir"] = run_dir
else:
    res = generate(cfg, out_root=OUTPUT_ROOT, scale=SCALE, return_internal=True)
    print("Wrote:", res["run_dir"])
print(f"generation time {time.time() - t0:.1f}s")
run_dir = res["run_dir"]
exports, gap, internal = res["exports"], res["gap_summary"], res["internal"]
files = {t: pd.read_csv(run_dir / f"{t}.csv") for t in TABLES}
files["label_gap_summary"] = pd.read_csv(run_dir / "label_gap_summary.csv")

## 3. Validation summary
Each check from spec 8.1 is run on the exported tables only. An empty problem list means PASS. Then: row counts, the 38 non-key features, and the SHA-256 hashes in the run manifest (they let anyone confirm a rerun is byte-identical).

In [ ]:
checks = validate_all(exports, cfg)
display(pd.DataFrame({"check": list(checks), "result": ["PASS" if not v else "FAIL" for v in checks.values()]}))
print("row counts:", {t: len(df) for t, df in exports.items()})
nk = non_key_features(); print(len(nk), "non-key features:", nk)
manifest = json.loads((run_dir / "manifest.json").read_text())
display(pd.Series(manifest["sha256"], name="sha256").to_frame())

## 4. Diagnostics (spec 7.3)

**Generator-side, uses internal truth. D3 must not do this.** The tables below use the true fraud flag (`y_true`) and the mechanism, which are not in any exported file. They exist to check that the generator behaves as designed.

In [ ]:
r_f = cfg["parameters"]["underlying_fraud_target"]["value"]
tx = internal.copy()
tx["month"] = tx["transaction_ts"].dt.strftime("%Y-%m")
def rate(g, col):
    t = tx.groupby(col)["y_true"].agg(n="size", frauds="sum")
    p = t.frauds / t.n; se = np.sqrt(p * (1 - p) / t.n)
    return t.assign(rate=p, lo=(p - 1.96 * se).clip(lower=0), hi=p + 1.96 * se)
n, k = len(tx), int(tx.y_true.sum()); p = k / n; se = np.sqrt(p * (1 - p) / n)
print(f"overall fraud rate {p:.5f}  95% CI [{max(p-1.96*se,0):.5f}, {p+1.96*se:.5f}]  target r_f = {r_f}  (n={n}, frauds={k})")
for col in ["month", "channel", "mechanism"]:
    print(f"\nby {col} (normal-approx 95% interval; coarse for small counts)")
    display(rate(tx, col))

**Merchant concentration.** Share of all fraud falling on the top 10% of merchants, and the Gini coefficient of fraud counts per merchant (0 = evenly spread, 1 = all in one merchant).

In [ ]:
mc = tx.groupby("merchant_id")["y_true"].sum().sort_values(ascending=False)
top = mc.head(max(1, int(round(0.1 * len(mc))))).sum() / mc.sum()
v = np.sort(mc.to_numpy(float)); i = np.arange(1, len(v) + 1)
gini = (2 * (i * v).sum() / (len(v) * v.sum())) - (len(v) + 1) / len(v)
print(f"merchants: {len(mc)}  top-10% share of fraud: {top:.3f}  Gini of fraud counts: {gini:.3f}")

**Amounts and velocity.** Amount percentiles for fraud versus non-fraud (overlap matters: fraud should not be trivially separable), and percentiles of the number of the card's transactions in the previous hour.

In [ ]:
q = [.01, .5, .9, .95, .99, 1.0]
amt = tx.groupby("y_true")["amount_php"].quantile(q).unstack(); amt.columns = ["p1", "p50", "p90", "p95", "p99", "max"]
display(amt.rename(index={False: "non-fraud", True: "fraud"}))
display(tx["tx_count_prev_1h"].quantile([.5, .9, .99, 1.0]).rename("tx_count_prev_1h").to_frame().T)

**Review coverage and alert-score yield.** Share of each day's transactions that were investigated, and, per decile of the latent risk score (the alert score itself is not kept by the generator), what share of investigated transactions were truly fraud.

In [ ]:
cov = tx.groupby(tx["transaction_ts"].dt.normalize())["r_investigated"].mean()
print("daily review coverage:"); display(cov.describe().to_frame("coverage").T)
inv = tx[tx["r_investigated"]].copy()
inv["decile"] = pd.qcut(inv["latent_risk_score"].rank(method="first"), 10, labels=range(1, 11))
display(inv.groupby("decile", observed=True)["y_true"].agg(n_investigated="size", true_fraud="sum", yield_share="mean"))

**Observed versus true fraud (gap).** From `label_gap_summary` overall: how much true fraud was found, and how many false confirmations occurred. Some labels are still pending at the end of the window.

In [ ]:
o = gap[gap.group == "overall"].iloc[0]
print(f"true fraud {o.n_actual_fraud}, observed fraud {o.n_observed_fraud}, true fraud observed {o.n_actual_fraud_observed}, "
      f"false confirmations {o.n_false_fraud_observed}")
print(f"recall of observed labels = {o.n_actual_fraud_observed / o.n_actual_fraud:.3f}")
print(res["info"]["review_status"], res["info"]["label_source"])

## 5. Analyst view
This is what a downstream analyst sees: the first rows of each exported table and the aggregate gap summary. **Only these files are D3 inputs**; D3 never loads row-level ground truth.

In [ ]:
for name, df in files.items():
    print(f"\n== {name}  ({len(df)} rows)"); display(df.head())

## 6. Limitations

- All parameter values are scenario assumptions for a runnable generator, not calibration evidence.
- Some vocabularies are still OPEN (placeholders pending approval).
- Card reissue and blocking after fraud are not simulated.
- Distance-from-home values are concentrated (limited geographic realism).
- Labels still pending at the end of the window are null, so observed-fraud counts are slightly understated at the tail.